# Phase 2B — Azure Leakage-Safe Data Preparation & Preprocessing

This notebook demonstrates and inspects the approved Phase 2B data preparation pipeline for the Microsoft Azure Predictive Maintenance (PdM) track in **FedMaint-AI**.

## Key Architectural Boundaries & Research Assumptions

1. **Prediction Unit & Temporal Horizon:**
   - Prediction unit: one machine at hourly prediction timestamp $t$.
   - Target label: binary $y = 1$ if at least one failure replacement episode occurs in $(t, t + 24\text{ hours}]$; $y = 0$ otherwise.
   - Episode collapse: multiple component failures recorded for the same machine at the exact same timestamp are collapsed into a single failure episode. Repeated timestamps at different hours remain distinct episodes.

2. **Observation History:**
   - Exactly 48 hourly readings in $[t - 48\text{h}, t)$, i.e. $t-48\text{h}, \dots, t-1\text{h}$.
   - No telemetry reading at $t$ or later is included in predictors (strict past-only, zero lookahead).
   - Primary features are telemetry-only (`volt`, `rotate`, `pressure`, `vibration`). No same-hour error or maintenance records are joined.

3. **Blackout Assumption:**
   - A candidate time $t$ is excluded if an episode occurs at $t$ or in the prior 48h ($[t - 48\text{h}, t]$ inclusive).
   - This is an explicit research blackout assumption representing post-failure/repair downtime and immediate recovery, not a physical failure onset.

4. **Temporal Splits (Half-Open $[T_{\text{start}}, T_{\text{end}})$):**
   - `train`: `[2015-01-01 06:00:00, 2015-09-01 06:00:00)`
   - `val`: `[2015-09-01 06:00:00, 2015-11-01 06:00:00)`
   - `test`: `[2015-11-01 06:00:00, 2016-01-01 06:00:00)`
   - Split membership rule: $t - 48\text{h} \ge T_{\text{start}}$ and $t < T_{\text{end}}$ and strict $t + 24\text{h} < T_{\text{end}}$. Neither history nor horizon ever crosses a split boundary.

5. **Client Partitioning & Held-Out Factory Isolation:**
   - Participating federated clients: `client_model2`, `client_model3`, `client_model4` (allowed splits: `train`, `val`, `test`).
   - Held-out client: `heldout_model1` (all 16 machines with model1). Model1 participates in **test** evaluation only. Any attempt to request train/val splits raises `HoldoutLeakageError`.

6. **Per-Window Normalization:**
   - Median and IQR computed strictly from the 48 past observations of that window.
   - Linear quartile interpolation (NumPy default $R-7$).
   - Exact zero scale fallback: 1.0 when $\text{IQR} == 0.0$.
   - Raw summaries (`median`, `iqr`, `scale`, `q25`, `q75`) are retained in every sample.
   - Zero cross-client or population fitting.

7. **Window Overlap Semantics:**
   - Consecutive hourly prediction windows for the same machine share 47 out of 48 observation hours.
   - A single failure episode generates up to 24 consecutive positive hourly windows. **Positive windows are NOT independent failure events.**

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root and src/ are in sys.path
current_path = Path.cwd().resolve()
if (current_path / "src").is_dir():
    repo_root = current_path
elif (current_path.parent / "src").is_dir():
    repo_root = current_path.parent
else:
    repo_root = current_path

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from fedmaint.data import (
    load_azure_preprocessing_config,
    load_pipeline_data,
    summarize_azure_pipeline,
    generate_azure_windows,
    get_client_partitions,
    load_machines,
    HoldoutLeakageError,
)

print(f"FedMaint-AI loaded from: {src_path}")

## 1. Inspect Frozen Phase 2B Configuration

We load `configs/azure_preprocessing.json` as the single source of truth.

In [ ]:
config = load_azure_preprocessing_config()
print(f"Pipeline: {config.pipeline_name} (Phase {config.phase}, v{config.version})")
print(f"History window: {config.history_hours} hours | Horizon: {config.horizon_hours} hours | Cadence: {config.cadence_hours} hour")
print(f"Primary sensors: {config.sensor_names}")
print(f"Participating models: {config.participating_models}")
print(f"Held-out model: {config.held_out_model}")
print("\nTemporal Splits:")
for split_name, bounds in config.splits.items():
    print(f"  - {split_name:5s}: [{bounds.start} to {bounds.end})")

## 2. Locate Azure Data Directory

Resolve the directory containing the validated Azure CSV files (`PdM_telemetry.csv`, etc.).

In [ ]:
data_dir = os.environ.get("AZURE_DATA_DIR")
if data_dir is None:
    candidates = [
        repo_root / "data" / "Azure Data",
        repo_root.parent / "data" / "Azure Data",
    ]
    for candidate in candidates:
        if candidate.is_dir() and (candidate / "PdM_telemetry.csv").is_file():
            data_dir = candidate
            break

if data_dir is not None:
    data_dir = Path(data_dir)
    print(f"Found Azure data at: {data_dir}")
else:
    print("Azure raw data files not found in standard paths.")
    print("Codex will execute this notebook in the feature repository where raw CSVs are available.")

## 3. Verify Client Partitioning and Holdout Isolation

Verify that the 100 machines are partitioned strictly by machine model, and that `heldout_model1` disallows train/val split queries.

In [ ]:
if data_dir is not None:
    machines = load_machines(data_dir)
    partitions = get_client_partitions(machines, config=config)
    print("Client Partitions:")
    for name, p in partitions.items():
        print(f"  - {name:15s}: model={p.model}, machines={len(p.machine_ids)}, held_out={p.is_held_out}, allowed_splits={p.allowed_splits}")
    
    # Verify HoldoutLeakageError guardrail
    heldout = partitions["heldout_model1"]
    try:
        heldout.validate_split_access("train")
        print("ERROR: Holdout leakage check failed!")
    except HoldoutLeakageError as exc:
        print(f"\nVerified Holdout Guardrail: {exc}")

## 4. Run Pipeline Summary on Real Data

When executed by Codex in the environment with raw data, this indexes telemetry into memory once (actual peak memory unmeasured) and computes the complete, inspectable counts table across all clients and splits without materializing unneeded feature arrays.

In [ ]:
pipeline_data = None
if data_dir is not None:
    print("Loading and indexing Azure PdM data into memory...")
    pipeline_data = load_pipeline_data(data_dir, config=config)
    
    print("Computing Phase 2B pipeline summary...")
    summary = summarize_azure_pipeline(pipeline_data=pipeline_data, config=config)
    
    print("\n========================================================================================")
    print("Phase 2B Azure Pipeline Summary Table")
    print("========================================================================================")
    header = f"{'Client':<16} {'Split':<6} {'Candidate':<10} {'Accepted':<10} {'Positives':<10} {'Negatives':<10} {'Prev %':<8} {'Episodes':<9}"
    print(header)
    print("-" * len(header))
    
    for client_name, c_data in summary["clients"].items():
        for split_name, s_data in c_data["splits"].items():
            print(
                f"{client_name:<16} {split_name:<6} "
                f"{s_data['candidate_windows']:<10} "
                f"{s_data['accepted_windows']:<10} "
                f"{s_data['positive_windows']:<10} "
                f"{s_data['negative_windows']:<10} "
                f"{s_data['positive_prevalence_pct']:<8.2f} "
                f"{s_data['total_episodes_in_split']:<9}"
            )
    print("========================================================================================\n")
    
    print("Rejection Breakdown & Episode Details:")
    for client_name, c_data in summary["clients"].items():
        print(f"\n[{client_name}] (model: {c_data['model']}, machines: {c_data['machine_count']})")
        for split_name, s_data in c_data["splits"].items():
            print(f"  Split: {split_name}")
            print(f"    - Rejections by reason: {s_data['rejections_by_reason']}")
            print(f"    - Labelable episodes: {s_data['labelable_episodes']} / {s_data['total_episodes_in_split']}")
            print(f"    - Unlabelable episodes: {s_data['unlabelable_episodes']}")
            print(f"    - Unlabelable breakdown: {s_data['unlabelable_breakdown']}")
else:
    print("Skipping execution: raw data files not present in staging workspace.")

## 5. Sample Inspection (Shape, Order, and Normalization)

Inspect the first accepted sample from `client_model2` in the `train` split. Note that the sample payload contains normalized features, binary target label, raw sensor summaries, and observation intervals; future event records are omitted.

In [ ]:
if pipeline_data is not None:
    sample_gen = generate_azure_windows(
        pipeline_data=pipeline_data,
        client_name="client_model2",
        split_name="train",
        config=config,
    )
    first_sample = next(sample_gen)
    
    print(f"Sample machineID: {first_sample.machine_id}")
    print(f"Prediction time t: {first_sample.prediction_time}")
    print(f"Observation window: [{first_sample.observation_start} to {first_sample.observation_end}] (48 hours)")
    print(f"Label horizon: ({first_sample.horizon_start} to {first_sample.horizon_end}] (24 hours)")
    print(f"Label y: {first_sample.label}")
    print(f"Feature matrix shape: ({len(first_sample.features)} steps, {len(first_sample.features[0])} sensors)")
    print(f"Sensor order: {first_sample.sensor_names}")
    print("\nRaw Sensor Summaries (retained in sample):")
    for s, stats in first_sample.raw_summaries.items():
        print(f"  - {s:10s}: median={stats['median']:.2f}, IQR={stats['iqr']:.2f}, scale={stats['scale']:.2f}")
    
    print("\nFirst normalized timestep features (volt, rotate, pressure, vibration):")
    print([round(v, 4) for v in first_sample.features[0]])